In [ ]:
import os
import shutil
from pathlib import Path

# 音频源目录
source_dir = "/home/you/workspace/son/crema-d-mirror/AudioWAV"
# 输出目标目录
target_dir = "/home/you/workspace/son/FastSpeech2/raw_data/CREMA-D"

# 文本缩写映射字典
text_map = {
    "IEO": "It's eleven o'clock.",
    "TIE": "That is exactly what happened.",
    "IOM": "I'm on my way to the meeting.",
    "IWW": "I wonder what this is about.",
    "TAI": "The airplane is almost full.",
    "MTI": "Maybe tomorrow it will be cold.",
    "IWL": "I would like a new alarm clock.",
    "ITH": "I think I have a doctor's appointment.",
    "DFA": "Don't forget a jacket.",
    "ITS": "I think I've seen this before.",
    "TSI": "The surface is slick.",
    "WSI": "We'll stop in a couple of minutes."
}

# 创建目标根目录
os.makedirs(target_dir, exist_ok=True)

# 遍历所有 wav 文件
for file_name in os.listdir(source_dir):
    if not file_name.lower().endswith(".wav"):
        continue

    # 文件名格式为: 1016_TIE_NEU_XX.wav
    parts = file_name.split('_')
    if len(parts) < 2:
        print(f"跳过无效文件名: {file_name}")
        continue

    speaker_id = parts[0]
    sentence_id = parts[1]

    text = text_map.get(sentence_id)
    if not text:
        print(f"未找到匹配文本: {file_name}")
        continue

    speaker_dir = os.path.join(target_dir, speaker_id)
    os.makedirs(speaker_dir, exist_ok=True)

    source_path = os.path.join(source_dir, file_name)
    target_wav_path = os.path.join(speaker_dir, file_name)
    target_lab_path = target_wav_path.replace('.wav', '.lab')

    # 拷贝音频
    shutil.copyfile(source_path, target_wav_path)

    # 写入 lab 文件
    with open(target_lab_path, 'w') as f:
        f.write(text)

print("处理完成！")


In [4]:
emotion2va = {
    "NEU": (4.0, 4.0),
    "HAP": (5.0, 7.0),
    "SAD": (3.5, 1.0),
    "ANG": (7.0, 1.0),
    "FEA": (5.5, 2.0),
    "DIS": (7.0, 3.0),
    # a v
}

def extract_emotion(wav_id: str):
    parts = wav_id.split("_")
    return parts[2] if len(parts) >= 3 else "UNK"

def estimate_va(emotion):
    return emotion2va.get(emotion, (4.0, 4.0))

def add_va_info(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as fin, \
        open(output_path, "w", encoding="utf-8") as fout:
        for line in fin:
            line = line.strip()
            if not line:
                continue

            fields = line.split("|")
            utt_id = fields[0]  # e.g., 1082_DFA_NEU_XX
            emotion = extract_emotion(utt_id)
            v, a = estimate_va(emotion)
            extended_line = f"{line}|{emotion}|{v}|{a}"
            fout.write(extended_line + "\n")

    print(f"✅ 已生成带VA信息的新文件：{output_path}")

input_path = "/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train_original.txt"
output_path = "/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train.txt"
add_va_info(input_path, output_path)
input_path = "/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/val_original.txt"
output_path = "/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/val.txt"
add_va_info(input_path, output_path)

✅ 已生成带VA信息的新文件：/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train.txt
✅ 已生成带VA信息的新文件：/home/kai/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/val.txt


In [ ]:
'''生成带所有信息的train.txt'''
import os
import torch
import types
import librosa
import torch.nn as nn
from tqdm import tqdm
from transformers import AutoModelForAudioClassification
from transformers.models.wav2vec2.modeling_wav2vec2 import (
    Wav2Vec2Model, Wav2Vec2PreTrainedModel
)

# === AVD Encoder Definitions ===

class ADV(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.dense = nn.Linear(config.hidden_size, config.hidden_size)
        self.out_proj = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, x):
        x = self.dense(x)
        x = torch.tanh(x)
        return self.out_proj(x)

class Dawn(Wav2Vec2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.wav2vec2 = Wav2Vec2Model(config)
        self.classifier = ADV(config)

    def forward(self, x):
        x -= x.mean(1, keepdim=True)
        variance = (x * x).mean(1, keepdim=True) + 1e-7
        x = self.wav2vec2(x / variance.sqrt())
        return self.classifier(x.last_hidden_state.mean(1))

def _forward(self, x):
    x = (x + self.config.mean) / self.config.std
    x = self.ssl_model(x, attention_mask=None).last_hidden_state
    h = self.pool_model.sap_linear(x).tanh()
    w = torch.matmul(h, self.pool_model.attention).softmax(1)
    mu = (x * w).sum(1)
    x = torch.cat([mu, ((x * x * w).sum(1) - mu * mu).clamp(min=1e-7).sqrt()], 1)
    return self.ser_model(x)

# === Coordinate Conversion ===

@torch.no_grad()
def avd_to_spherical_raw(avd: torch.Tensor, M: torch.Tensor, eps=1e-8):
    avd_shifted = avd - M  # [B, 3]
    r_raw = torch.norm(avd_shifted, dim=1)
    # IQR 归一化
    q1 = r_raw.quantile(0.25)
    q3 = r_raw.quantile(0.75)
    iqr = q3 - q1
    r_min = q1 - 1.5 * iqr
    r_max = q3 + 1.5 * iqr
    r_norm = ((r_raw - r_min) / (r_max - r_min + eps)).clamp(0, 1)
    a_p, v_p, d_p = avd_shifted[:, 0], avd_shifted[:, 1], avd_shifted[:, 2]
    theta = torch.acos(torch.clamp(d_p / (r_raw + eps), -1 + eps, 1 - eps))  # θ: [0, π]
    phi = torch.atan2(v_p, a_p)  # φ: [-π, π]
    return r_raw, r_norm, theta, phi

# === Main Processing ===

def process_txt(input_txt, output_txt, wav_root):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    base = AutoModelForAudioClassification.from_pretrained(
        '3loi/SER-Odyssey-Baseline-WavLM-Multi-Attributes',
        trust_remote_code=True).to(device).eval()
    base.forward = types.MethodType(_forward, base)

    dawn = Dawn.from_pretrained(
        'audeering/wav2vec2-large-robust-12-ft-emotion-msp-dim'
    ).to(device).eval()

    def wav2small(x):
        return 0.5 * dawn(x) + 0.5 * base(x)

    avd_list, is_neutral_list, line_list, emotion_list = [], [], [], []

    with open(input_txt, "r") as fin:
        for line in tqdm(fin):
            parts = line.strip().split("|")
            utt_id = parts[0]
            speaker = parts[1]
            emotion_tag = utt_id.split("_")[2]
            is_neutral = "NEU" in emotion_tag
            wav_path = os.path.join(wav_root, speaker, utt_id + ".wav")

            if not os.path.exists(wav_path):
                print(f"[WARN] Missing: {wav_path}")
                continue

            signal = torch.from_numpy(librosa.load(wav_path, sr=16000)[0])[None, :]
            pred = wav2small(signal.to(device)).cpu()
            a, v, d = pred[0, 0], pred[0, 2], pred[0, 1]

            emotion_list.append(emotion_tag)
            avd_list.append([a.item(), v.item(), d.item()])
            is_neutral_list.append(is_neutral)
            line_list.append(line.strip())

    avd_tensor = torch.tensor(avd_list)  # [B, 3]
    neutral_mask = torch.tensor(is_neutral_list)

    M = avd_tensor[neutral_mask].mean(dim=0, keepdim=True)  # 中性中心

    r_raws, r_norms, thetas, phis = avd_to_spherical_raw(avd_tensor, M)

    with open(output_txt, "w") as fout:
        for line, emotion, (a, v, d), r_norm, theta, phi in zip(line_list, emotion_list, avd_tensor, r_norms, thetas, phis):
            fout.write(f"{line}|{emotion}|{a:.6f}|{v:.6f}|{d:.6f}|{r_norm.item():.6f}|{theta.item():.6f}|{phi.item():.6f}\n")


# === Entry Point ===

if __name__ == "__main__":
    input_path = "/home/you/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train_original.txt"
    output_path = "/home/you/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train_emosphere.txt"
    wav_root = "/home/you/workspace/son/FastSpeech2/raw_data/CREMA-D/"

    process_txt(input_path, output_path, wav_root)

    input_path = "/home/you/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/val_original.txt"
    output_path = "/home/you/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/val_emosphere.txt"

    process_txt(input_path, output_path, wav_root)


In [1]:
import collections

import torch

def compute_avg_theta_phi(train_txt):
    emo_dict = collections.defaultdict(list)

    with open(train_txt, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("|")
            if len(parts) < 2:
                continue
            emotion = parts[4]  # 假设情绪在第5列（索引4）
            try:
                theta = float(parts[-2])  # 倒数第二列
                phi = float(parts[-1])    # 倒数第一列
                emo_dict[emotion].append((theta, phi))
            except:
                continue  # 可能是 header 或空行

    print("=== 平均 θ 和 φ ===")
    for emo, values in emo_dict.items():
        values_tensor = torch.tensor(values)
        theta_avg = values_tensor[:, 0].mean().item()
        phi_avg = values_tensor[:, 1].mean().item()
        print(f"{emo:8s}  θ = {theta_avg:.4f},  φ = {phi_avg:.4f}")


# 使用示例

compute_avg_theta_phi("/home/you/workspace/son/FastSpeech2/preprocessed_data/CREMA-D/train.txt")


=== 平均 θ 和 φ ===
SAD       θ = 2.1294,  φ = -1.2915
FEA       θ = 1.5596,  φ = -0.5081
ANG       θ = 0.9223,  φ = -0.0775
HAP       θ = 1.1065,  φ = 0.3045
DIS       θ = 1.5408,  φ = -0.6281
NEU       θ = 1.6091,  φ = -0.1555
